# 🧮 T4 x2 で max_model_len が載るかの計算（GPU 不要）

config.json と実ファイルサイズから KV cache と残 VRAM を見積る。


In [ ]:
# T4 x2 で max_model_len が載るかを config から計算（GPU 不要）
import json
import os

ROOT = ("/kaggle/input/datasets/yamahei21python/"
        "qwen38-27b-uncensored-hauhaucs-awq-mtp")
cfg = json.load(open(os.path.join(ROOT, "config.json")))

print("architectures        :", cfg.get("architectures"))
print("model_type           :", cfg.get("model_type"))
print("num_hidden_layers    :", cfg.get("num_hidden_layers"))
print("hidden_size          :", cfg.get("hidden_size"))
print("num_attention_heads  :", cfg.get("num_attention_heads"))
print("num_key_value_heads  :", cfg.get("num_key_value_heads"))
print("head_dim             :", cfg.get("head_dim"))
print("max_position_emb     :", cfg.get("max_position_embeddings"))
print("torch_dtype          :", cfg.get("torch_dtype"))
print("quantization_config  :", json.dumps(cfg.get("quantization_config"), ensure_ascii=False))
print("rope_scaling         :", cfg.get("rope_scaling"))
print("tie_word_embeddings  :", cfg.get("tie_word_embeddings"))

L = cfg["num_hidden_layers"]
H = cfg["hidden_size"]
NH = cfg["num_attention_heads"]
NKV = cfg.get("num_key_value_heads") or NH
HD = cfg.get("head_dim") or (H // NH)
VOCAB = cfg.get("vocab_size")
print("\nderived: head_dim", HD, "kv_heads", NKV, "vocab", VOCAB)

# --- 重みサイズ（実ファイル）---
def sz(name):
    p = os.path.join(ROOT, name)
    return os.path.getsize(p) if os.path.isfile(p) else 0

main = sz("model.safetensors")
mtp = sz("model-mtp.safetensors")
print(f"\nweights: main {main/1e9:.2f} GB / mtp {mtp/1e9:.2f} GB")

# --- KV cache（fp16）---
kv_per_token = 2 * L * NKV * HD * 2          # K+V, 2 bytes
print(f"KV per token (fp16): {kv_per_token/1024**2:.3f} MiB")
print(f"KV for 32768 tokens: {kv_per_token*32768/1e9:.2f} GB (TP=2 で /2)")
print(f"KV for 16384 tokens: {kv_per_token*16384/1e9:.2f} GB (TP=2 で /2)")

# --- T4 x2 の見積り ---
GPU_MEM = 15.36           # GiB (15360 MiB)
UTIL = 0.90
N = 2
budget = GPU_MEM * UTIL * N
weight_per_gpu = (main + mtp) / N
print(f"\nTP=2: budget {budget:.2f} GiB / weights {weight_per_gpu:.2f} GiB "
      f"→ KV+act {budget - weight_per_gpu:.2f} GiB")
for ml in (32768, 16384, 8192, 4096):
    need = kv_per_token * ml / N
    free = budget - weight_per_gpu - need
    verdict = "OK" if free > 1.0 else "NG"
    print(f"  max_model_len={ml:6d}: KV {need:6.2f} GiB, 残 {free:6.2f} GiB -> {verdict}")

# --- 量子化の妥当性（vLLM の awq は 4bit / group 128 / zp 有無）---
q = cfg.get("quantization_config") or {}
print("\nquant check: bits=", q.get("bits"), "group_size=", q.get("group_size"),
      "zero_point=", q.get("zero_point"), "sym=", q.get("sym"))

# --- 理論重みサイズとの比較（AWQ 4bit なら ~14-15GB 期待）---
if VOCAB:
    print(f"embed/lm_head fp16 概算: {2*VOCAB*H*2/1e9:.2f} GB (tie={cfg.get('tie_word_embeddings')})")
